# Lesson 12 - Скорочення історії чату за допомогою блокнота агента

Цей нотатник демонструє, як керувати контекстом у довгих розмовах за допомогою Microsoft Agent Framework. У міру зростання розмов кількість токенів збільшується — зрештою перевищуючи вікно контексту моделі. Ми вирішуємо цю проблему за допомогою **патерну резюмування контексту** та **блокнота агента** для постійної пам’яті.

## Чого ви навчитеся:
1. **Чому важливо керувати контекстом**: розуміння обмежень токенів і вікон контексту
2. **Агенти з урахуванням контексту**: створення агентів, які керують власним контекстом розмов
3. **Патерн резюмування контексту**: використання інструментів для стиснення історії розмови
4. **Блокнот агента**: постійна пам’ять, що зберігається після скорочення контексту

## Попередні вимоги:
- Налаштування Azure OpenAI з конфігурованими змінними оточення
- Розуміння базових концепцій агентів з попередніх уроків


## Налаштування


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Чому важливо керувати контекстом

Кожна LLM має обмежене **вікно контексту** — максимальну кількість токенів, яку вона може обробити за один запит. У міру розвитку багатокрокової розмови:

- **Кількість токенів зростає лінійно** з кожним повідомленням користувача та відповіддю асистента.
- **Токени запиту домінують у вартості**, оскільки вся історія повторно надсилається на кожному кроці.
- Зрештою розмова **перевищує вікно контексту**, і модель або обрізає текст, або видає помилку.

### Стратегії керування контекстом

| Стратегія | Як працює | Компроміс |
|---|---|---|
| **Обрізання** | Відкидає найстаріші повідомлення | Втрачається ранній контекст |
| **Підсумовування** | Стискає старіші повідомлення у резюме | Деякі деталі втрачаються, але ключові моменти зберігаються |
| **Чернетка / Зовнішня пам'ять** | Зберігає ключові факти поза розмовою | Вимагає викликів інструментів, але зберігається при будь-якому скороченні |

У цій записній книжці ми поєднуємо **підсумовування** з **інструментом чернетки**, щоб агент міг підтримувати безперервність навіть при стислому історичному контексті.


## Створення агента, який враховує контекст


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Імітація тривалої розмови

Давайте пройдемося по багатокроковій розмові, щоб побачити, як накопичується контекст. Агент має зберігати ключові деталі (уподобання, бюджет, дати подорожі) протягом усієї розмови та демонструвати безперервність.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Зверніть увагу, як агент зберігає контекст із попередніх кроків — він знає про Японію, суші, храми, фотографію, бюджет у $3000, соло-подорож і період у квітні. У короткій розмові це працює добре, але з ростом розмови відправляти всю історію стає затратним.

Продовжимо розмову з більшою кількістю кроків, щоб побачити накопичення контексту:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Шаблон підсумовування контексту

У міру розвитку розмови ми можемо використовувати **інструмент підсумовування**, щоб стиснути накопичений контекст у компактний формат. Аґент викликає цей інструмент для фіксації ключових уподобань, щоб навіть якщо старіші повідомлення видаляються, важлива інформація залишалася збереженою.

Цей шаблон є будівельним блоком для більш складного скорочення історії:
1. Аґент визначає ключові факти з розмови
2. Викликає інструмент підсумовування для їх збереження
3. Старіші повідомлення можна безпечно видаляти, бо підсумок захоплює суттєве

Нижче ми визначаємо інструмент `summarize_preferences`, який аґент може викликати, щоб зафіксувати компактний підсумок того, що він дізнався.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Резюме

У цьому уроці ви навчилися керувати контекстом у довготривалих розмовах агента за допомогою Microsoft Agent Framework:

### Ключові поняття
- **Вікна контексту є обмеженими** — кожен токен в історії розмови коштує грошей і враховується до ліміту.
- **Інструменти узагальнення** дозволяють агенту стискати накопичений контекст у компактні резюме, зменшуючи використання токенів, при цьому зберігаючи важливу інформацію.
- **Блокноти агента** надають постійну зовнішню пам’ять, яка зберігається навіть при зменшенні розміру розмови.

### Що ви створили
- **Агента, обізнаного про контекст**, який підтримує безперервність у багатокрокових розмовах
- **Інструмент узагальнення** (`summarize_preferences`), що записує ключові деталі користувача в компактному форматі
- **Багатокрокову розмову**, що демонструє збереження контексту та обробку змін

### Реальні застосування
- **Боти служби підтримки клієнтів**: запам’ятовують уподобання користувачів під час тривалих сесій підтримки
- **Персональні помічники**: відстежують поточні проєкти без повторного пояснення контексту
- **Навчальні репетитори**: підтримують прогрес учня протягом багатьох взаємодій

### Наступні кроки
- Реалізувати повноцінний інструмент блокнота з файловим збереженням
- Додати автоматичне скорочення історії після узагальнення
- Поєднати з векторними базами даних для семантичного пошуку у пам’яті
- Створити агентів, які можуть відновлювати розмови через кілька днів з повним контекстом


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено з використанням сервісу автоматичного перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується звертатись до професійного людського перекладача. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
